# 🔑 Solutions: Lesson 6: Data Leakage
**Try the problems yourself first!** Looking at a solution before struggling is like watching someone else do push-ups.
Every solution below is self-contained and has a short explanation of *why* it works.

In [1]:
import sys
sys.path.append("../utils")            # lets us import course_utils.py and agri_prep.py

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from course_utils import *             # load_raw_farm, profile, setup_plots, ...

setup_plots()
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 140)
np.random.seed(42)


from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.feature_selection import SelectKBest, f_classif, f_regression
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge
fe = build_farm_features(load_lesson_clean())
prices = load_prices()

---
### Problem 6.1 &nbsp; ⭐ &nbsp; Spot the leak with a correlation scan

From the lesson-clean data, create `implied_yield = total_revenue_ksh / (farm_size_ha * 1000 * market_price_ksh_kg)`.

1. Compute the correlation of `implied_yield` with `yield_t_ha` (pairwise complete rows).
2. Explain why this proves `total_revenue_ksh` cannot be used as a feature when predicting yield.
3. Name two *other* columns of the dataset that would not be known at planting time.

In [2]:
fe["implied_yield"] = fe["total_revenue_ksh"] / (fe["farm_size_ha"] * 1000 * fe["market_price_ksh_kg"])
r = fe["implied_yield"].corr(fe["yield_t_ha"])
print(f"correlation(implied_yield, yield) = {r:.3f}")
print("Others unknown at planting: harvest_date, market_price_ksh_kg (set at sale) - and, strictly, seasonal rainfall/temperature/pest pressure.")

correlation(implied_yield, yield) = 0.995
Others unknown at planting: harvest_date, market_price_ksh_kg (set at sale) - and, strictly, seasonal rainfall/temperature/pest pressure.


**💬 Why this works:** A correlation near 1 between a *formula involving a feature* and the target means the feature is a disguised copy of the answer. The model would 'predict' yield by simply undoing the multiplication: useless for farms that haven't harvested yet. Always ask: *would I have this number on prediction day?*

---
### Problem 6.2 &nbsp; ⭐⭐ &nbsp; Which scaler statistics did the test set influence?

Split the `rainfall_mm` column (drop NaN) 80/20 with `random_state=0`.

1. Fit a `StandardScaler` on **all** the data and on the **train** part only. Transform the test set with each.
2. Print the learned mean and the mean/std of the scaled test set in both cases.
3. How large is the difference for plain scaling on ~2,000 rows? In which situations would you expect it to matter much more?

In [3]:
r = fe[["rainfall_mm"]].dropna()
train, test = train_test_split(r, test_size=0.2, random_state=0)

leaky  = StandardScaler().fit(r)           # sees test data!
honest = StandardScaler().fit(train)       # train only

for name, sc in (("leaky (fit on ALL)", leaky), ("honest (fit on TRAIN)", honest)):
    t = sc.transform(test)
    print(f"{name:22s} learned mean {sc.mean_[0]:7.2f} | test mean {t.mean():+.3f}, test std {t.std():.3f}")

leaky (fit on ALL)     learned mean  660.54 | test mean -0.064, test std 0.979
honest (fit on TRAIN)  learned mean  663.46 | test mean -0.080, test std 0.974


**💬 Why this works:** The two scalers learn slightly different means, and the leaky one lets the test rows pull the statistics toward themselves. With ~2,000 rows the numerical difference is tiny: plain scaling is a *mild* leak. It grows with small samples, extreme values in the test set, or when the step uses the target (selection, target encoding). The habit costs nothing, and it is what protects you when the stakes are higher.

---
### Problem 6.3 &nbsp; ⭐⭐ &nbsp; How does leakage depend on sample size?

Repeat the 'selection on pure noise' experiment with random data:

- **A:** n = 100 samples, p = 2000 features, select k = 10.
- **B:** n = 5000 samples, p = 2000 features, select k = 10 (use `cv=3` to keep it fast).

For each, compare the cross-validated accuracy of (1) selecting on *all* data first (leaky) and (2) a `Pipeline(SelectKBest, LogisticRegression)` (honest). Average over 5 random repetitions. Where is leakage worst, and why?

In [4]:
def experiment(n, p, k=10, reps=5, cv=5, seed=0):
    rng = np.random.default_rng(seed)
    leaky, honest = [], []
    for _ in range(reps):
        X = rng.normal(size=(n, p)); y = rng.integers(0, 2, n)
        Xs = SelectKBest(f_classif, k=k).fit_transform(X, y)
        leaky.append(cross_val_score(LogisticRegression(max_iter=1000), Xs, y, cv=cv).mean())
        pipe = make_pipeline(SelectKBest(f_classif, k=k), LogisticRegression(max_iter=1000))
        honest.append(cross_val_score(pipe, X, y, cv=cv).mean())
    return float(round(np.mean(leaky), 3)), float(round(np.mean(honest), 3))

print("A (n=100,  p=2000): leaky, honest =", experiment(100, 2000))
print("B (n=5000, p=2000): leaky, honest =", experiment(5000, 2000, cv=3, reps=3))

A (n=100,  p=2000): leaky, honest = (0.81, 0.536)


B (n=5000, p=2000): leaky, honest = (0.546, 0.493)


**💬 Why this works:** With few rows, *some* of the 2000 random columns will match the random labels by sheer luck, and selecting on all rows hunts them down: then cross-validation 'confirms' them because those same rows were used to choose them. With plenty of rows, chance correlations are tiny and the leak nearly vanishes. Leakage bites hardest on small, wide datasets.

---
### Problem 6.4 &nbsp; ⭐⭐ &nbsp; The centered-window trap on bean prices

Take **Beans** prices (sort by date; forward-fill gaps). Build `target = next day's price`. Features: today's price plus either a **trailing** 7-day mean or a **centered** 7-day mean.

Fit linear regressions on data before 2023 and test on 2023. Compare the MAE of: naive (tomorrow = today), trailing, centered. Which looks best, and why is it suspicious?

In [5]:
b = prices[prices["crop"] == "Beans"].sort_values("date").reset_index(drop=True)
b["price"] = b["price_ksh_kg"].ffill()
b["trailing7"] = b["price"].rolling(7).mean()
b["centered7"] = b["price"].rolling(7, center=True).mean()
b["target"]    = b["price"].shift(-1)
b = b.dropna(subset=["price", "trailing7", "centered7", "target"])

train, test = b[b["date"] < "2023-01-01"], b[b["date"] >= "2023-01-01"]
mae = {"naive (tomorrow = today)": (test["target"] - test["price"]).abs().mean()}
for name, cols in (("trailing window", ["price", "trailing7"]), ("centered window (LEAK)", ["price", "centered7"])):
    m = LinearRegression().fit(train[cols], train["target"])
    mae[name] = (test["target"] - m.predict(test[cols])).abs().mean()
print(pd.Series(mae).round(3).to_string())

naive (tomorrow = today)    2.751
trailing window             3.317
centered window (LEAK)      2.345


**💬 Why this works:** The centered window quietly averages in tomorrow's price, so the model 'forecasts' the future by looking at it. Its MAE beats both the naive baseline and the honest trailing window: a *too-good-to-be-true* improvement that vanishes in production, where tomorrow isn't available. When a fancy feature beats a simple baseline by a lot, audit its construction first.

---
### Problem 6.5 &nbsp; ⭐⭐⭐ &nbsp; Repair a leaky workflow

A colleague's code predicts yield for a **small sample** (200 random farms) using 15 real features plus 300 junk columns. It (1) selects the 15 'best' features using *all* rows and then (2) cross-validates.

1. Build the leaky version and report its 5-fold CV R².
2. Rewrite it as one `Pipeline` (median imputation → `SelectKBest(f_regression, k=15)` → `Ridge`) cross-validated honestly, and report its R².
3. By how much was the colleague's score inflated?

In [6]:
from sklearn.impute import SimpleImputer

s = fe.sample(200, random_state=0)
real = ["rainfall_mm", "avg_temp_c", "soil_ph", "nitrogen_ppm", "organic_matter_pct", "fertilizer_kg_ha", "irrigation",
        "rain_gap_sq", "temp_gap_sq", "ph_gap_sq", "log_fert", "pest_ord", "exp_ord", "npk_total", "moisture_index"]
rng = np.random.default_rng(0)
junk = pd.DataFrame(rng.normal(size=(200, 300)), index=s.index, columns=[f"junk_{i}" for i in range(300)])
X = pd.concat([s[real], junk], axis=1)
y = np.log(s["yield_t_ha"])

# --- LEAKY: impute + select on ALL rows, then cross-validate
X_all = SimpleImputer(strategy="median").fit_transform(X)
X_sel = SelectKBest(f_regression, k=15).fit_transform(X_all, y)
leaky = cross_val_score(Ridge(), X_sel, y, cv=5, scoring="r2").mean()

# --- HONEST: everything inside the pipeline (re-fitted on each training fold)
pipe = make_pipeline(SimpleImputer(strategy="median"), SelectKBest(f_regression, k=15), Ridge())
honest = cross_val_score(pipe, X, y, cv=5, scoring="r2").mean()

print(f"leaky  R² = {leaky:.3f}\nhonest R² = {honest:.3f}\ninflation = {leaky - honest:+.3f}")

leaky  R² = 0.546
honest R² = 0.436
inflation = +0.111


**💬 Why this works:** In the honest pipeline each CV fold chooses its 15 features from its own training rows only, then is scored on rows that took no part in the choice. The leaky version lets every row vote on the selection, including the rows later used for scoring. Expect the leaky number to be higher; the size of the gap depends on how many junk columns got lucky.